In [1265]:
from random import random, sample, randrange
from icecream import ic

In [1266]:
N = 10

In [1267]:
OBJECTS = {n for n in range(N)}
SETS = tuple(frozenset(sample(range(N), s+1)) for s in range(N))
COSTS = tuple(10*random()+(s*random())**2 for s in range(N))



## Solution: array of bools


In [1268]:
sol = tuple(sample([True, False], N, counts=(N,N)))
sol

(True, False, False, False, False, True, True, False, False, True)

In [1269]:
from typing import Sequence

def fitness_aob(solution: Sequence[bool]) -> float:
    weight = sum(COSTS[n] for n in range(N) if solution[n])
    check = frozenset.union(*(SETS[n] for n in range(N) if solution[n]))
    return -weight


In [1270]:
def tweak_aob(solution: Sequence[bool]) -> Sequence[bool]:
    i = randrange(N)
    new_solution = list(solution)
    new_solution[i] = not new_solution[i]
    check = frozenset().union(*(SETS[n] for n in range(N) if new_solution[n]))
    if check == OBJECTS:
        return tuple(new_solution)
    return tuple(solution)

In [1271]:
steps = 0
steps_without_improvement = 0
while True:
    new_sol = tweak_aob(sol)
    steps += 1
    if fitness_aob(new_sol) > fitness_aob(sol):
        steps_without_improvement = 0
        sol = new_sol
        ic(steps, fitness_aob(sol))
    else: 
        steps_without_improvement += 1
        if steps_without_improvement > 50:
            break

ic| steps: 2, fitness_aob(sol): -32.195027726230485
ic| steps: 6, fitness_aob(sol): -31.07450984539016
ic| steps: 9, fitness_aob(sol): -20.087454035250364


## Solution: set of sets


In [1272]:
XCOST = {SETS[i]: COSTS[i] for i in range(N)}
sol = set(sample(SETS, N//2))
sol

{frozenset({2, 3}),
 frozenset({0, 3, 8}),
 frozenset({0, 1, 4, 5, 6, 7, 8, 9}),
 frozenset({0, 2, 4, 6, 9}),
 frozenset({0, 1, 2, 3, 4, 5, 6, 7, 8, 9})}

In [1273]:
def fitness_sos(solution: set[frozenset]) -> float:
    weight = sum(XCOST[s] for s in solution)
    check = frozenset.union(*solution)
    
    return -weight

In [1274]:
def tweak_sos(solution: set[frozenset]) -> set[frozenset]:
    solution = set(solution)
    i = randrange(N)
    if SETS[i] in solution:
        new_solution = solution.copy()
        new_solution.remove(SETS[i])
    else:
        new_solution = solution.copy()
        new_solution.add(SETS[i])

    check = frozenset().union(*new_solution)
    if check == OBJECTS:
        return new_solution
    return solution

In [1275]:
steps = 0
steps_without_improvement = 0
while True:
    new_sol = tweak_sos(sol)
    steps += 1
    if fitness_sos(new_sol) > fitness_sos(sol):
        steps_without_improvement = 0
        sol = new_sol
        ic(steps, fitness_sos(sol))
    else: 
        steps_without_improvement += 1
        if steps_without_improvement > 100:
            break
    

ic| steps: 2, fitness_sos(sol): -31.34364434962928
ic| steps: 6, fitness_sos(sol): -27.484090650428755
ic| steps: 7, fitness_sos(sol): -21.417930123107833
ic| steps: 26, fitness_sos(sol): -20.087454035250364
